In [8]:
from sentence_transformers import SentenceTransformer
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

model_name = 'sentence-transformers/paraphrase-xlm-r-multilingual-v1'
model = SentenceTransformer(model_name)

In [9]:
documents = [
    "This a list which containing sample documents.",
    "Keywords are important for keyword-based search.",
    "Document analysis involves extracting keywords.",
    "Keyword-based search relies on sparse embeddings.",
    "Understsnding document structure aids in keyword extraction.",
    "Efficient keyword extraction enhances search accuracy.",
    "Semacntic similarity improves document retrieval performance.",
    "Machine learning techniques can optimize keyword extraction methods."
]
len(documents)

query = "Natural language processing techniques enhances keyword extraction efficiency."

In [10]:
def tokenize(text):
    return re.findall(r'\w+', text.lower())

In [11]:
embeddings = model.encode(documents)
query_embedding = model.encode(query)

similarities = cosine_similarity([query_embedding], embeddings)
sorted_indices = np.argsort(similarities[0])[::-1]

ranked_docs = [(documents[i], similarities[0][i]) for i in sorted_indices]
top_4_docs = [doc[0] for doc in ranked_docs[:4]]
top_4_docs

['Machine learning techniques can optimize keyword extraction methods.',
 'Efficient keyword extraction enhances search accuracy.',
 'Understsnding document structure aids in keyword extraction.',
 'Document analysis involves extracting keywords.']

In [12]:
tokenized_documents = [tokenize(doc) for doc in top_4_docs]
tokenized_query = tokenize(query)

tokenized_documents, tokenized_query

([['machine',
   'learning',
   'techniques',
   'can',
   'optimize',
   'keyword',
   'extraction',
   'methods'],
  ['efficient', 'keyword', 'extraction', 'enhances', 'search', 'accuracy'],
  ['understsnding',
   'document',
   'structure',
   'aids',
   'in',
   'keyword',
   'extraction'],
  ['document', 'analysis', 'involves', 'extracting', 'keywords']],
 ['natural',
  'language',
  'processing',
  'techniques',
  'enhances',
  'keyword',
  'extraction',
  'efficiency'])

In [13]:
bm25 = BM25Okapi(tokenized_documents)

scores = bm25.get_scores(tokenized_query)
scores

array([1.06000097, 1.21203299, 0.3119808 , 0.        ])

In [14]:
sorted_indices = np.argsort(scores)[::-1]

reranked_docs_bm25 = [(top_4_docs[i], scores[i]) for i in sorted_indices]
for doc, score in reranked_docs_bm25:
    print(f"Document: {doc}, BM25 Score: {score:.4f}")

Document: Efficient keyword extraction enhances search accuracy., BM25 Score: 1.2120
Document: Machine learning techniques can optimize keyword extraction methods., BM25 Score: 1.0600
Document: Understsnding document structure aids in keyword extraction., BM25 Score: 0.3120
Document: Document analysis involves extracting keywords., BM25 Score: 0.0000
